In [ ]:
import os
from pathlib import Path
import pandas as pd
import requests

# Look for .env in the parent directory if your notebook is in a subfolder (e.g., notebooks/)
env_path = Path("../../.env") if Path("../../.env").exists() else Path(".env")
for line in env_path.read_text().splitlines() if env_path.exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
API_KEY = os.environ.get("RAWTREE_API_KEY")
TEAM = os.environ.get("TEAM_ID")

if not API_KEY:
    raise ValueError("RAWTREE_API_KEY is missing! Check your .env file location.")

http = requests.Session()
http.headers["Authorization"] = f"Bearer {API_KEY}"

In [9]:
def query(sql: str) -> pd.DataFrame:
    """Helper function to execute SQL queries against the RawTree API and return a DataFrame."""
    r = http.post(f"{API}/v1/query", json={"sql": sql}, timeout=600)
    r.raise_for_status()
    data = r.json().get("data", [])
    return pd.DataFrame(data)

print("Fetching all prediction data sources from RawTree...")

Fetching all prediction data sources from RawTree...


In [10]:
# All-time best sellers ranking (derived from historical purchases)
sales_sql = """
SELECT product_id, count() AS n
FROM shop.history_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY n DESC
"""
best_df = query(sales_sql)
BEST_SELLERS: list[str] = best_df["product_id"].tolist() if not best_df.empty else []

# Current out-of-stock items (recommending these makes an answer invalid)
stock_sql = """
SELECT product_id::String AS product_id
FROM shop.catalog_updates
WHERE change::String IN ('out_of_stock', 'back_in_stock')
GROUP BY product_id
HAVING argMax(change::String, ts::DateTime64(3)) = 'out_of_stock'
"""
stock_df = query(stock_sql)
OUT_OF_STOCK: set[str] = set(stock_df["product_id"]) if not stock_df.empty else set()



In [11]:
# Co-views: For every product, the top 20 products most viewed in the same sessions
coviews_sql = """
SELECT a.product_id AS p, b.product_id AS q, count() AS n
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS b
  ON a.session_id = b.session_id
WHERE a.product_id != b.product_id
GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p
"""
coviews_df = query(coviews_sql)
COVIEWS_DICT: dict[str, list[str]] = (
    coviews_df.groupby("p").q.apply(list).to_dict() if not coviews_df.empty else {}
)


In [12]:

# Active incoming recommendation requests queue
requests_sql = """
SELECT request_id, ts, user_id, context, recent_views
FROM shop.requests
ORDER BY ts DESC
LIMIT 100
"""
RECENT_REQUESTS: pd.DataFrame = query(requests_sql)



In [13]:

# Historical session logs (contains impressions, clicks, propensity scores)[cite: 1]
history_sessions_sql = """
SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
FROM shop.history_sessions
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_SESSIONS: pd.DataFrame = query(history_sessions_sql)

# Granular historical event logs (individual views and purchases for matrix factorization / item similarity)
history_events_sql = """
SELECT session_id, user_id, event_type, product_id, ts
FROM shop.history_events
ORDER BY ts DESC
LIMIT 10000
"""
HISTORY_EVENTS: pd.DataFrame = query(history_events_sql)


In [28]:
catalog = query("SELECT * FROM shop.catalog")
feedback_df = query(f"SELECT * FROM {TEAM}.feedback ORDER BY ts DESC LIMIT 1000")

HTTPError: 400 Client Error: Bad Request for url: https://api.rawtree.com/v1/query

In [14]:
# Summary of loaded variables
if __name__ == "__main__":
    print("\n--- Data Loaded Successfully ---")
    print(f"• BEST_SELLERS: {len(BEST_SELLERS)} products loaded")
    print(f"• OUT_OF_STOCK: {len(OUT_OF_STOCK)} items currently unavailable")
    print(f"• COVIEWS_DICT: {len(COVIEWS_DICT)} products have co-view maps")
    print(f"• RECENT_REQUESTS: {len(RECENT_REQUESTS)} live request rows fetched")
    print(f"• HISTORY_SESSIONS: {len(HISTORY_SESSIONS)} session rows fetched")
    print(f"• HISTORY_EVENTS: {len(HISTORY_EVENTS)} event rows fetched")


--- Data Loaded Successfully ---
• BEST_SELLERS: 4809 products loaded
• OUT_OF_STOCK: 36 items currently unavailable
• COVIEWS_DICT: 5000 products have co-view maps
• RECENT_REQUESTS: 100 live request rows fetched
• HISTORY_SESSIONS: 10000 session rows fetched
• HISTORY_EVENTS: 10000 event rows fetched


---

In [ ]:
bestSellers = pd.DataFrame(BEST_SELLERS)
outOfStock = pd.DataFrame(OUT_OF_STOCK)
coviewsDict = COVIEWS_DICT
recentRequests = pd.DataFrame(RECENT_REQUESTS)
historySessions = pd.DataFrame(HISTORY_SESSIONS)
historyEvents = pd.DataFrame(HISTORY_EVENTS)
catalog = catalog



In [27]:
catalog.head()

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards
